# Regional and Route Analysis

## Goal

1. How are facilities and routes distributed geographically?
2. Does delivery performance (on-time, detention) differ by facility?
3. Do individual facilities stand out, or is the spread consistent with chance?
4. Which lanes/routes are the most and least profitable?
5. How does revenue relate to route distance and rate structure?

## Data Basis

**Via pipeline** (`clean_regional_analysis.csv`, one row per load):
`loads`, `routes`, `trips` and the pickup/delivery events (pivoted into `pickup_*` / 
`delivery_*` columns), plus the delivery metrics and an estimated fuel cost per load 
(`fuel_cost`, aggregated per trip from `fuel_purchases`).

**Loaded directly in this notebook:**
- `facilities`: facility locations and types
- `delivery_events`: raw events, one row per event (for the facility analysis)
- `routes`, `fuel_purchases`, `trips`: raw tables for the data quality checks
- `maintenance_records`: for the cost estimate in section 6 (not loaded yet)

## 1. Load Data (via Pipeline)

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
sns.set_theme(style="darkgrid")

from delivery_pipeline import run_pipeline, PROCESSED_DATA_PATH_REGIONAL, RAW_DATA_DIR

logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

clean_df = run_pipeline(
    output_path=PROCESSED_DATA_PATH_REGIONAL,
    include_routes=True,
    include_fleet=False,
    include_costs=True
    )

print(f"Loads used for analysis: {len(clean_df)}")

fuel_df = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
facility_df = pd.read_csv(RAW_DATA_DIR / "facilities.csv")
routes_df = pd.read_csv(RAW_DATA_DIR / "routes.csv")
trips_df = pd.read_csv(RAW_DATA_DIR / "trips.csv")
delivery_df = pd.read_csv(RAW_DATA_DIR / "delivery_events.csv")


### 1.1 Verification of the fuel cost merge

In [ ]:
display(fuel_df["trip_id"].value_counts().describe())
display((fuel_df["total_cost"]<=0).value_counts())
display(fuel_df["trip_id"].isna().sum())
display(fuel_df["total_cost"].isna().sum())
display(trips_df["trip_id"].isin(fuel_df["trip_id"]).mean())
display(clean_df[["fuel_cost", "fuel_gallons_purchased", "n_fuel_purchases"]].head())

In [ ]:
fuel_sum = fuel_df["total_cost"].sum().round(2)
fuel_sum_merge = clean_df["fuel_cost"].sum().round(2)
diff = abs(fuel_sum-fuel_sum_merge).round(2)
removed = fuel_df.loc[~fuel_df["trip_id"].isin(clean_df["trip_id"]), "total_cost"].sum().round(2)
price = (clean_df["fuel_cost"].sum()/clean_df["fuel_gallons_purchased"].sum()).round(3)
avg_price = fuel_df["price_per_gallon"].mean().round(3)
has_fuel = clean_df["fuel_cost"].notna()

print(f"Sum of fuel_df: {fuel_sum}\nSum of clean_df: {fuel_sum_merge}\nDifference: {diff}\nRemoved values: {removed}")
print(f"Price per gallon (calculated): {price}, Price per gallon (fuel_df): {avg_price}")
print(clean_df.loc[has_fuel, ["fuel_gallons_purchased", "fuel_gallons_used"]].sum())

*The fuel cost merge is consistent: 90.1 % of loads have fuel data, and the difference between the raw and merged cost sums (574,953.83) equals exactly the fuel cost of the loads removed by the pipeline. The calculated price per gallon (3.899) matches the raw average (3.898). On loads with fuel data, about 43 % more gallons were purchased (24.4 M) than used (17.1 M).*

## 2. Data Quality Checks
### 2.1 Do all facility_id / route_id references resolve?

In [ ]:
unique_fac = facility_df["facility_id"].is_unique
unique_routes = routes_df["route_id"].is_unique
print(f"facility_id values are unique: {unique_fac}\nroute_id values are unique: {unique_routes}")

In [ ]:
def check_reference(relation, child, column, parent, parent_key):
    """Count rows, missing keys and orphans for one foreign key."""
    keys = child[column].dropna()
    return {
        "relation": relation,
        "rows": len(child),
        "missing_keys": child[column].isna().sum(),
        "orphans": (~keys.isin(parent[parent_key])).sum(),
    }

results = pd.DataFrame([
    check_reference("loads.route_id -> routes", clean_df, "route_id", routes_df, "route_id"),
    check_reference("events.facility_id -> facilities", delivery_df, "facility_id", facility_df, "facility_id"),
    check_reference("loads.pickup_facility_id -> facilities", clean_df, "pickup_facility_id", facility_df, "facility_id"),
    check_reference("loads.delivery_facility_id -> facilities", clean_df, "delivery_facility_id", facility_df, "facility_id"),
])
results["orphan_share"] = results["orphans"] / results["rows"]
display(results)

*All `route_id` references (84,924 loads), `facility_id` references (170,820 events) and the `pickup_facility_id` / `delivery_facility_id` columns of the loads (84,924 each) resolve; there are no orphans and no missing keys.*

### 2.2 Are location_city / location_state pairs plausible?

In [ ]:
sources = {
    "delivery_events": (delivery_df, "location_city", "location_state"),
    "fuel_purchases": (fuel_df, "location_city", "location_state"),
    "facilities": (facility_df, "city", "state"),
}
rows = []
for name, (df, city_col, state_col) in sources.items():
    rows.append({
        "table": name,
        "pairs": len(df[[city_col, state_col]].drop_duplicates()),
        "cities": df[city_col].nunique(),
        "max_states_per_city": df.groupby(city_col)[state_col].nunique().max(),
    })
display(pd.DataFrame(rows))

In [ ]:
merged = delivery_df.merge(facility_df[["facility_id", "city", "state"]], on="facility_id", how="left")
ok = (merged["location_city"] == merged["city"]) & (merged["location_state"] == merged["state"])
print(f"Events matching their facility: {ok.mean():.2%} ({ok.sum()} of {len(ok)})")

delivery_df.groupby("facility_id")["location_city"].nunique().describe()   

In [ ]:
# Chance baseline: how often would an event's city equal its facility's city
# if event cities were assigned independently of the facility?
# Compared with the observed match rate above (city and state): every city match
# is also a state match here, so the city-only rate is the same (3.45 %).

p_event = delivery_df["location_city"].value_counts(normalize=True)
p_fac = merged["city"].value_counts(normalize=True)
print(f"Match expected by chance (city): {(p_event * p_fac).sum():.2%}")
print(f"Observed city-only match: {(merged['location_city'] == merged['city']).mean():.2%}")

### Data Quality Note

* `delivery_events`: each of the 20 cities maps to exactly one state (internally plausible), but only 3.45 % of events match the city/state of their facility, about what chance gives (3.50 %). Every facility has events in all 20 cities.
* `fuel_purchases`: 575 city/state pairs, every city paired with 23 states, so the fields are effectively random (confirms notebook 3).
* **Decision**: location fields of events and fuel purchases are not used. Regions come from facilities and routes.

## 3. Facility and Route Overview
### 3.1 Where are the facilities located, and what do routes look like?

In [ ]:
# TODO: facilities by state/city; route distance distribution

In [ ]:
# TODO: optional - map using geocoordinates

*Interpretation*

## 4. Delivery Performance by Facility
### 4.1 Does on-time rate / detention differ by facility?

In [ ]:
# TODO: groupby facility_id -> on_time_flag mean, detention_minutes mean, n events

In [ ]:
# TODO chart(s)

### 4.2 Is that spread larger than expected by chance?

In [ ]:
# Permutation test (same approach as notebook 3, section 5.2), applied to facilities

In [ ]:
# TODO: plot shuffled distribution, mark observed value

*Interpretation*

### 4.3 Does on-time rate differ by event type (pickup vs. delivery)?

In [ ]:
# TODO: on_time_flag and detention_minutes by event_type

*TODO: interpretation - this was a large, clear difference in the exploratory check; confirm and quantify it here.*

## 5. Regional Delivery Performance
### 5.1 Does delay/duration differ by region (state)?

In [ ]:
# TODO: groupby state -> delay rate, average duration (revisit notebook 1 finding with distance normalized, see 5.2)

### 5.2 Does the regional duration difference remain after accounting for distance?

In [ ]:
# TODO: compute hours per 100 miles (or average speed) per trip
# TODO: compare regions on this normalized metric instead of raw duration

*TODO: interpretation - notebook 1 found duration varies a lot by region (11-40h); check whether this is mostly explained by route distance.*

## 6. Route / Lane Profitability
### 6.1 What is the revenue per route?

In [ ]:
# TODO: aggregate loads.revenue by route_id

### 6.2 What are the approximate costs per route?

In [ ]:
# TODO: estimate fuel cost per route (via trips -> fuel_purchases, aggregated per trip then per route)
# TODO: optionally add a share of maintenance cost per mile (from notebook 3 / maintenance_records)
# Prüfen, ob die Lücken zufällig sind. Vergleiche die Distanz von Trips mit und ohne Fuel-Eintrag, 
# z. B. mit groupby auf trips_df["trip_id"].isin(fuel_df["trip_id"]). Fehlen Einträge vor allem bei 
# kurzen Trips, ist das ein Befund für die Data Quality Note in 6.2.
# Lücken auf Zufälligkeit prüfen: gehört in Abschnitt 6.2, dort, wo du die Kosten den Routen 
# zuordnest und die Data Quality Note schreibst. Dort entscheidest du, wie du mit den 10 % fehlenden Fuel-Kosten umgehst.

### Data Quality Note

*TODO: note the assumptions/approximations made when allocating fuel and maintenance cost to routes.*

### 6.3 Which lanes are most and least profitable?

In [ ]:
# TODO: margin = revenue - estimated cost, per route; rank; chart
# Rechne in 6.3 mit dem Mittelwert über Loads mit Fuel-Daten, und gib die Abdeckung je Route an.
# Margin pro Load rechnen und dann mitteln: gehört in 6.3. Hier merkst du dir nur, dass du fehlende 
# fuel_cost nicht mit 0 füllst, sonst bleibt die Pipeline so, wie sie ist.

*TODO: interpretation.*

## 7. Conclusion

*TODO: summarize facility-level findings, the pickup-vs-delivery difference, the distance-adjusted regional comparison, and route profitability; link to notebook 5.*